In [32]:
# Importa a biblioteca pandas, usada para manipular tabelas de dados (como se fosse um Excel no código)
import pandas as pd
# Importa a biblioteca base de aprendizado de máquina
import sklearn
# Importa o tradutor que transforma textos/categorias em matrizes numéricas para o modelo entender
from sklearn.feature_extraction import DictVectorizer
# Importa o algoritmo de Regressão Linear, que vai aprender e fazer as previsões matemáticas
from sklearn.linear_model import LinearRegression
# Importa a ferramenta que calcula a margem de erro das previsões do modelo (Root Mean Squared Error)
from sklearn.metrics import mean_squared_error

In [2]:
pd.__version__

'1.4.2'

In [3]:
# Baixa e lê o arquivo de dados dos táxis de janeiro de 2024 e salva na variável 'df' (tabela principal)
df = pd.read_parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet")

In [4]:
!pip install pyarrow

In [5]:
# (Opcional) Mostra as 5 primeiras linhas da tabela na tela (funciona melhor se for a última linha da célula no Jupyter)
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [9]:
sklearn.__version__

'1.0.2'

In [30]:
# Subtrai a hora final da hora inicial, converte tudo para segundos e divide por 60 para ter o resultado em minutos.
df["duration"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

In [31]:
# Cria a coluna duration
df["duration"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60
# Filtra a tabela para manter apenas viagens normais (entre 1 e 60 minutos)
df = df[(df.duration >= 1) & (df.duration <= 60)]

In [25]:
# Define quais colunas são categorias (locais) e quais são numéricas (distância)
categorical = ["PULocationID", "DOLocationID"]
numerical = ["trip_distance"]

In [26]:
# Isso garante que o modelo entenda que "1" e "2" são bairros diferentes, e não grandezas matemáticas.
df[categorical] = df[categorical].astype(str)

In [27]:
# Converte as colunas selecionadas em uma lista de "dicionários" (pares de nome da coluna e seu valor)
train_dicts = df[categorical + numerical].to_dict(orient="records")
# Inicia o tradutor de categorias
dv = DictVectorizer()
# O tradutor lê os dicionários e cria o 'X_train' (as pistas traduzidas em formato de matriz matemática)
X_train = dv.fit_transform(train_dicts)
# Define quem é o 'y_train' (o gabarito). Extraímos apenas os valores da coluna 'duration'
y_train = df["duration"].values

In [28]:
# Inicia o algoritmo de Inteligência Artificial
model = LinearRegression()
# O modelo estuda as pistas (X_train) e o gabarito (y_train) para encontrar o padrão matemático
model.fit(X_train, y_train)
# Como não tem os dados de fevereiro (validação) carregados neste script, 
# vou testar o modelo fazendo ele prever os próprios dados de janeiro que ele acabou de estudar.
y_pred = model.predict(X_train)
# Calcula a média de minutos que o modelo errou, comparando o gabarito real (y_train) com a sua adivinhação (y_pred)
rmse = mean_squared_error(y_train, y_pred, squared=False)

In [29]:
# Exibe a margem de erro na tela
print(f"The error medio is the {rmse} minutes")

The error medio is the 7.946010390556888 minutes
